### 1. Receita Total da Base de Filmes
**Pergunta de Negócio:** Qual é a receita total (em R$) somada de todos os filmes da base?

> **Objetivo:** Calcular a soma consolidada de faturamento de todas as produções convertidas para moeda local (BRL) na tabela fato da camada Gold.

In [0]:
%sql

SHOW TABLES IN cinedata_gold;

database,tableName,isTemporary
cinedata_gold,bridge_movie_company,false
cinedata_gold,bridge_movie_genre,false
cinedata_gold,bridge_movie_person,false
cinedata_gold,dim_companies,false
cinedata_gold,dim_genres,false
cinedata_gold,dim_movies,false
cinedata_gold,dim_people,false
cinedata_gold,dim_reviews,false
cinedata_gold,fact_movies_performance,false
cinedata_gold,gold_genai_movies_context,false


In [0]:
%sql
SELECT 
    ROUND(SUM(receita_brl), 2) AS receita_total_brl
FROM cinedata_gold.fact_movies_performance;

receita_total_brl
604265308391.64


### 2. Top 5 Filmes por Popularidade
**Pergunta de Negócio:** Quais são os 5 filmes com maior popularidade? Mostre título e valor de popularidade.

> **Objetivo:** Identificar os títulos mais populares da base realizando o JOIN entre a tabela fato de performance e a dimensão de filmes.

In [0]:
%sql
SELECT 
    d.titulo,
    f.popularidade
FROM cinedata_gold.fact_movies_performance f
JOIN cinedata_gold.dim_movies d ON f.sk_movie_id = d.sk_movie_id
ORDER BY f.popularidade DESC
LIMIT 5;

titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
La Fellinette,2020.0
The Fear Footage 2: Curse of the Tape,2019.0
wwe survivor series 2018,2018.0


### 3. Distribuição de Filmes por Gênero
**Pergunta de Negócio:** Quantos filmes cada gênero possui? Liste do maior para o menor volume.

> **Objetivo:** Mapear o volume de títulos categorizados por gênero cinematográfico utilizando a tabela ponte (bridge) N:N de relacionamentos.

In [0]:
%sql
SELECT 
    g.nome_genero,
    COUNT(DISTINCT bg.sk_movie_id) AS total_filmes
FROM cinedata_gold.bridge_movie_genre bg
JOIN cinedata_gold.dim_genres g ON bg.sk_genre_id = g.sk_genre_id
GROUP BY g.nome_genero
ORDER BY total_filmes DESC;

nome_genero,total_filmes
Drama,30457
Documentary,18612
Comedy,17348
Thriller,9424
Horror,9163
Romance,6996
Action,5541
Crime,4320
Animation,4159
Tv Movie,3676


### 4. Ranking dos 10 Filmes de Maior Receita
**Pergunta de Negócio:** Para os 10 filmes de maior receita, mostre título, receita (em US$ e R$) e a posição de cada um no ranking (`RANK()`).

> **Objetivo:** Aplicar função de janela (Window Function `RANK()`) para ranquear as 10 maiores bilheterias globais em ambas as moedas.

In [0]:
%sql
WITH ranking_receita AS (
    SELECT 
        d.titulo,
        f.receita_usd,
        f.receita_brl,
        RANK() OVER (ORDER BY f.receita_usd DESC) AS posicao_ranking
    FROM cinedata_gold.fact_movies_performance f
    JOIN cinedata_gold.dim_movies d ON f.sk_movie_id = d.sk_movie_id
)
SELECT 
    posicao_ranking,
    titulo,
    ROUND(receita_usd, 2) AS receita_usd,
    ROUND(receita_brl, 2) AS receita_brl
FROM ranking_receita
WHERE posicao_ranking <= 10
ORDER BY posicao_ranking;

posicao_ranking,titulo,receita_usd,receita_brl
1,Avengers: Endgame,2800000000.00,11094720000.00
2,Avatar: The Way of Water,2320250281.00,12390136500.54
3,AVENGERS: INFINITY WAR,2052415039.00,7190430847.63
4,spider-man: no way home,1921847111.00,10977782882.74
5,The Lion King,1663075401.00,6227552146.58
6,Top Gun: Maverick,1488732821.00,7160804869.01
7,Barbie,1428545028.00,6856159007.38
8,The Super Mario Bros. Movie,1355725263.00,6838413799.10
9,Black Panther,1349926083.00,4429782441.36
10,Star Wars: The Last Jedi,1332698830.00,4401904235.49


### 5. Ator com Maior Número de Participações nos Últimos 2 Anos
**Pergunta de Negócio:** Qual ator teve a maior quantidade de participações nos filmes lançados nos últimos 2 anos*?

> **Objetivo:** Filtrar atores (`papel_categoria = 'actor'`) com base na janela temporal dinâmica de 24 meses contados a partir da maior data de lançamento válida da base.

In [0]:
%sql
WITH max_data AS (
    SELECT MAX(data_lancamento) AS max_date 
    FROM cinedata_gold.dim_movies 
    WHERE data_lancamento <= CURRENT_DATE()
)
SELECT 
    p.nome_pessoa AS nome_ator,
    COUNT(DISTINCT bp.sk_movie_id) AS total_participacoes
FROM cinedata_gold.bridge_movie_person bp
JOIN cinedata_gold.dim_people p ON bp.sk_person_id = p.sk_person_id
JOIN cinedata_gold.dim_movies d ON bp.sk_movie_id = d.sk_movie_id
CROSS JOIN max_data m
WHERE (LOWER(p.tipo_pessoa) LIKE '%actor%' OR LOWER(p.tipo_pessoa) LIKE '%ator%')
  AND d.data_lancamento >= ADD_MONTHS(m.max_date, -24)
  AND d.data_lancamento <= m.max_date
GROUP BY p.nome_pessoa
ORDER BY total_participacoes DESC
LIMIT 1;

nome_ator,total_participacoes
Kevin Hart,66


### 6. Produtora Mais Lucrativa nos Últimos 5 Anos
**Pergunta de Negócio:** Qual a produtora de filmes teve o maior Lucro nos últimos 5 anos*?

> **Objetivo:** Agrupar o lucro acumulado ($\text{Receita} - \text{Orçamento}$) por empresa produtora na janela dinâmica dos últimos 60 meses a partir do marco temporal da base.

In [0]:
%sql
WITH max_data AS (
    SELECT MAX(data_lancamento) AS max_date 
    FROM cinedata_gold.dim_movies 
    WHERE data_lancamento <= CURRENT_DATE()
)
SELECT 
    e.nome_produtora AS produtora,
    ROUND(SUM(f.lucro_usd), 2) AS lucro_total_usd,
    ROUND(SUM(f.lucro_brl), 2) AS lucro_total_brl
FROM cinedata_gold.bridge_movie_company be
JOIN cinedata_gold.dim_companies e ON be.sk_company_id = e.sk_company_id
JOIN cinedata_gold.fact_movies_performance f ON be.sk_movie_id = f.sk_movie_id
JOIN cinedata_gold.dim_movies d ON f.sk_movie_id = d.sk_movie_id
CROSS JOIN max_data m
WHERE d.data_lancamento >= ADD_MONTHS(m.max_date, -60)
  AND d.data_lancamento <= m.max_date
GROUP BY e.nome_produtora
ORDER BY lucro_total_usd DESC
LIMIT 1;

produtora,lucro_total_usd,lucro_total_brl
Universal Pictures,5298113092.00,26835791008.34
